READ THE DATA FROM EXCEL

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt

# Load Bike Stations Data
bike_stations = pd.read_excel('shared bikes stations_2024.xlsx')

# Load Bike Trips Data
bike_trips = pd.read_csv('Abonnement Modus données trajets.csv')

# Load PT Ridership Data French Zone
pt_ridership_french = pd.read_excel('mn_montees-par-arret-par-ligne-par-tranchehoraire(AutoRecovered).xlsx') 

#pt_ridership_french data set
# This dataset concerns 2 lines which are out of TPG Network. 
# Number of users who board a vehicle per stop, per line and per day for M and N lines. 
# These values apply to the whole network without distinction between the Swiss zone and the French zone. 
# These are the values recorded in the sensors at the top of the doors of each vehicle, successively consolidated in our systems. 
# Raw values are "improved" by quality control algorithms, which can fill in missing data (usually less than 2%) and balance races in the event of differences between boarding and alighting passengers.

pt_ridership_general = pd.read_excel('montees-par-arret-par-ligne.xlsx')

# Convert timestamps
bike_trips['trip_started_at_utc'] = pd.to_datetime(bike_trips['trip_started_at_utc'], utc=True)
bike_trips['trip_ended_at_utc'] = pd.to_datetime(bike_trips['trip_ended_at_utc'], utc=True)

pt_ridership_french['Date'] = pd.to_datetime(pt_ridership_french['Date'])
pt_ridership_general['Date'] = pd.to_datetime(pt_ridership_general['Date'])

# Drop any completely empty rows
bike_stations.dropna(how='all', inplace=True)
bike_trips.dropna(how='all', inplace=True)
pt_ridership_french.dropna(how='all', inplace=True)
pt_ridership_general.dropna(how='all', inplace=True)

# Display missing value summaries
print("Missing Values in Bike Stations:\n", bike_stations.isnull().sum())
print("\nMissing Values in Bike Trips:\n", bike_trips.isnull().sum())
print("\nMissing Values in PT Ridership French Zone:\n", pt_ridership_french.isnull().sum())
print("\nMissing Values in PT Ridership General:\n", pt_ridership_general.isnull().sum())

bike_trips_clean = bike_trips.dropna(subset=["latitude_start", "longitude_start", "latitude_end", "longitude_end"])
# --- Check for Missing Coordinate Values ---
missing_coords = bike_trips_clean[["latitude_start", "longitude_start", "latitude_end", "longitude_end"]].isnull().sum()
if missing_coords.sum() > 0:
    print("Warning: There are missing coordinate values in bike_trips_clean:")
    print(missing_coords)
else:
    print("No missing coordinate values found in bike_trips_clean.")


#Convert to geodataframe

# Convert PT Ridership to GeoDataFrame
gdf_pt_ridership = gpd.GeoDataFrame(
    pt_ridership_french,
    geometry=gpd.points_from_xy(
        x=pt_ridership_french['Stop Longtitudes'], 
        y=pt_ridership_french['Stop Latitudes']
    ),
    crs="EPSG:4326"
)

# Convert Bike Stations to GeoDataFrame
gdf_bike_stations = gpd.GeoDataFrame(
    bike_stations,
    geometry=gpd.points_from_xy(
        x=bike_stations['longitude'], 
        y=bike_stations['latitude']
    ),
    crs="EPSG:4326"
)

gdf_pt_ridership_general =gpd.GeoDataFrame(
    pt_ridership_general,
    geometry=gpd.points_from_xy(
        x=pt_ridership_general['Stop longtitudes'], 
        y=pt_ridership_general['Stop latitudes']
    ),
    crs="EPSG:4326"
) 

# Step 1: Get unique stops from each dataset
unique_pt_ridership = gdf_pt_ridership.drop_duplicates(subset='geometry')
unique_pt_ridership_general = gdf_pt_ridership_general.drop_duplicates(subset='geometry')

# Step 2: Find stops from pt_ridership_general that are NOT in pt_ridership
unique_general_only = unique_pt_ridership_general[
    ~unique_pt_ridership_general.geometry.isin(unique_pt_ridership.geometry)
]

unique_bike_stations = gdf_bike_stations.drop_duplicates(subset='geometry')

# Create GeoDataFrames for origin and destination points
bike_trips_clean.loc[:, 'origin_bikes'] = [Point(xy) for xy in zip(bike_trips_clean['longitude_start'], bike_trips_clean['latitude_start'])]
bike_trips_clean.loc[:, 'destination_bikes'] = [Point(xy) for xy in zip(bike_trips_clean['longitude_end'], bike_trips_clean['latitude_end'])]

display(unique_bike_stations.head(10))


display(unique_general_only.head(10))


display(bike_trips_clean.head(10))




CLUSTER THE NODES AS SUPER NODES BASED ON CLUSTERING RADIUS USING DBSCAN (density based clusterin)

THIS CREATES THE ORIGIN AND DESTINATION POINTS THAT CAN INVOLVE BIKE STATIONS AND/OR PT STATIONS. 

SUPER NODES ALSO HAVE THE INFORMATION OF WHICH PT LINES ARE STOPPING IN THAT SUPER NODE.

CREATE A LEAFLET MAP FOR VISUALIZATION

In [ ]:
from Codes.map_and_hub_creator import create_hub_map_and_dataframe

m, hub_df_sample = create_hub_map_and_dataframe(unique_bike_stations, unique_general_only, 120)


m


SAMPLE THE NETWORK FOR CREATING SMALLER INSTANCES

In [ ]:

import geopandas as gpd
from shapely.geometry import Point
import pickle


# Define the city center coordinates in EPSG:3857 (x, y)
hub_id_values = [1, 14, 81]
city_centers  = []
target_radii = [1000, 1500, 2000, 2500]  # Create a buffer (e.g., 3000 meters radius)
zones = []
hub_df_samples = []

for id in hub_id_values:
    for rad in target_radii:
            city_center = hub_df_sample_120.loc[hub_df_sample_120['hub_id'] == id, 'geometry'].iloc[0]
            city_centers.append(city_center)
            zone = city_center.buffer(rad)
            zones.append(zone)
            # Filter the hub_df to only include nodes within the zone
            hub_df_sample_120 = gpd.GeoDataFrame(hub_df_sample_120, geometry='geometry', crs="EPSG:3857")
            hub_df_sample_run = hub_df_sample_120[hub_df_sample_120.geometry.within(zone)]
            hub_df_samples.append(hub_df_sample_run)
            print(f"Original number of hubs: {len(hub_df_sample_120)}")
            print(f"Number of hubs in target zone: {len(hub_df_sample_run)}")


BELOW PROCESS EXTRACTS BIKE ITINERARIES FROM GENEVA DATA TO ADD IT AS A SCALING FACTOR WHEN WE CREATE DEMAND IN INSTANCE GENERATOR

In [ ]:
import pandas as pd

bike_trips_df = matched_bike_trips_bike_stations

# Assume bike_trips_df is your original trips DataFrame.
# Filter for Fall trips in time slot "06h-09h".
#sub_df = bike_trips_df[(bike_trips_df['season'] == "Fall") & (bike_trips_df['time_slot'] == "06h-09h")]

# Pivot so that each trip_id has one row with origin and destination.
sub_df_pivot = bike_trips_df.pivot(index='trip_id', columns='point_type', 
                            values='matched bike station name').reset_index()

# Build mapping from individual bike station names to hub_id.
# Here, hub_df has a column "bike_stations" where each entry is a list of station names.
station_to_hub = {}
for idx, row in hub_df.iterrows():
    for station in row["bike_stations"]:
        station_to_hub[station] = row["hub_id"]

# Map the station names in the pivoted DataFrame to hub IDs.
sub_df_pivot['origin_hub'] = sub_df_pivot['origin'].map(station_to_hub)
sub_df_pivot['destination_hub'] = sub_df_pivot['destination'].map(station_to_hub)
print(len(sub_df_pivot))
# Drop rows where the mapping failed.
sub_df_pivot = sub_df_pivot.dropna(subset=['origin_hub', 'destination_hub'])
print(len(sub_df_pivot))

# Aggregate OD trips by grouping by origin_hub and destination_hub.
od_counts = sub_df_pivot.groupby(['origin_hub', 'destination_hub']).size().reset_index(name='trip_count')

print(od_counts)

with open('od_counts.pkl', 'wb') as f:
    pickle.dump(od_counts,f)
import pandas as pd

bike_trips_df = matched_bike_trips_bike_stations

# Assume bike_trips_df is your original trips DataFrame.
# Filter for Fall trips in time slot "06h-09h".
#sub_df = bike_trips_df[(bike_trips_df['season'] == "Fall") & (bike_trips_df['time_slot'] == "06h-09h")]

# Pivot so that each trip_id has one row with origin and destination.
sub_df_pivot = bike_trips_df.pivot(index='trip_id', columns='point_type', 
                            values='matched bike station name').reset_index()

# Build mapping from individual bike station names to hub_id.
# Here, hub_df has a column "bike_stations" where each entry is a list of station names.
station_to_hub = {}
for idx, row in hub_df.iterrows():
    for station in row["bike_stations"]:
        station_to_hub[station] = row["hub_id"]

# Map the station names in the pivoted DataFrame to hub IDs.
sub_df_pivot['origin_hub'] = sub_df_pivot['origin'].map(station_to_hub)
sub_df_pivot['destination_hub'] = sub_df_pivot['destination'].map(station_to_hub)
print(len(sub_df_pivot))
# Drop rows where the mapping failed.
sub_df_pivot = sub_df_pivot.dropna(subset=['origin_hub', 'destination_hub'])
print(len(sub_df_pivot))

# Aggregate OD trips by grouping by origin_hub and destination_hub.
od_counts = sub_df_pivot.groupby(['origin_hub', 'destination_hub']).size().reset_index(name='trip_count')

print(od_counts)

with open('od_counts.pkl', 'wb') as f:
    pickle.dump(od_counts,f)


INSTANCE GENERATOR USING REAL NETWORK DATA

- generate_instance_real only randomizes choice coefficients, capacity in bike stations, inventory level in bike stations and O-D demand. 
- process is:
    - get super node data from above clustering process
    - create bike and PT edges between each O-D pair based on mode of transportation (i.e. create all possible single leg alternatives between O-D)
    - create 1 transfer alternatives
    - create 2 transfer alternatives
    - create the indicator function based on the alternatives
    - assign demand and capacities to each super node
    - save the instance as a json and txt

In [ ]:
import random
import os
from Codes.CPPMTO_instance_generator_v2 import generate_instance_real

# Generate the instance

# Set a random seed for reproducibility (optional)
seed_number=42
random.seed(seed_number)

# Price Inputs
M = 4                       # assign upper bound for the price.

beta_cost_range = (0.8, 1)          # cost sensitivity
beta_time_range = (0.5, 0.75)       # time sensitivity
beta_transfer_range = (0.5, 0.75)   # transfer sensitivity
beta_buffer_range = (0.5, 0.75)          # for car trips


# Capacity constraints

capacity_range = (100, 150)           # capacity for each bike station
inventory_range = (30, 35)          # how many bikes actually available

# =====================================================
# 2. Create a folder to store the generated instances
# =====================================================

Data_folder=r"C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\toy model" # Define the folder path to store the generated instances
folder_path = os.makedirs(Data_folder, exist_ok=True)
folder_path = Data_folder

json_paths = []

for hub in hub_df_samples:
    json_file_path, txt_file_path = generate_instance_real(folder_path, seed_number, hub, od_counts,
                           beta_cost_range, beta_time_range, beta_transfer_range, beta_buffer_range,
                           capacity_range, inventory_range, 
                           fixed_pt_price_value=4, bike_cost_operator=0.5, ridership_weight=2,
                           max_disp_alts=7, min_demand=10, max_demand=70, max_travel_time=80,
                           min_od_distance=500, bike_max_distance=3000, transfer_buffer_range=(1,10),
                           use_fixed_beta_values=True, 
                           fixed_beta_cost = 0.9, fixed_beta_time = 0.6, fixed_beta_transfer = 0.4, fixed_beta_buffer = 0.2                           
                           )
    json_paths.append(json_file_path)


SOLVE THE INSTANCES

In [ ]:
from Codes.CPPMTO_optimizer1 import solve_bilevel_model

save_folder = ... # Define the folder path to store the solutions



solution_paths = []

#Loop over the generated instances and solve the bilevel model

for json_file_path in json_paths:
    _,_, solution_file = solve_bilevel_model(json_file_path, save_folder, M=4, verbose=False, verbose2=False)
    solution_paths.append(solution_file)

print("Solutions are saved in the folder:", save_folder)


DISPLAY RESULTS

In [ ]:
# =====================================================
# 4. Display Dataframe for Solved Instances
# =====================================================

import pandas as pd
import numpy as np
import json
import os
import glob

# Define your base folder
solution_folder = r'C:\Users\mkoseogl\Desktop\Geneva Data\test_instance_real\toy model\sols'

# Get all JSON files with "solution" in their name
solution_paths_real_samples = glob.glob(os.path.join(solution_folder, '*solution*.json'))
    

#Store results from all solutions
all_summaries_real_samples = []

# Loop through all generated solutions
for solution_file in solution_paths_real_samples:
    # Ensure the solution file exists
    if os.path.exists(solution_file):
        with open(solution_file, "r") as f:
            data = json.load(f)
        solution_time = round(data["Solution Time (s)"], 2)  # Extract solution time
        Obj_Value= round(data["Objective Value"], 2)         # Extract objective value
        Gap = round(data["Gap (%)"], 2)              # Extract gap value
        df = pd.DataFrame(data["instances"])

        # Extract instance name
        instance_name = os.path.basename(solution_file).replace("_solution.json", "")

        # Compute key statistics
        num_od_pairs = df["OD"].nunique()  # Count unique OD pairs
        total_demand = round(df["Flow (q)"].sum(), 2)  # Total flow (demand)
        total_flow_car = round(df[df["Type"] == "Car"]["Flow (q)"].sum(), 2)  # Total flow for Car
        total_flow_pt_only = round(df[df["Type"] == "PT-Only"]["Flow (q)"].sum(), 2)  # Total flow for PT-Only
        total_flow_pt_bike = round(df[(df["Type"] == "PT+Bike") & (df["Flow (q)"] >= 1)]["Flow (q)"].sum(), 2)  # Total flow for PT+Bike

        # Compute Average Bike Price for PT+Bike where Flow ≥ 1
        avg_bike_price_pt_bike = df[(df["Type"] == "PT+Bike")& (df["Flow (q)"] >= 1)]["Bike Price"].mean()
        avg_bike_price_pt_bike_total = df[(df["Type"] == "PT+Bike") & (df["nu"] == 1)]["Bike Price"].mean()
        avg_num_transfers_pt_bike = df[(df["Type"] == "PT+Bike") & (df["Flow (q)"] >= 1)]["Transfers"].mean()
        avg_num_transfers_pt_only = df[(df["Type"] == "PT-Only") & (df["Flow (q)"] >= 1)]["Transfers"].mean()
        
        # **NEW METRICS**
        total_pt_only_arcs = df[df["Type"] == "PT-Only"].shape[0]  # Total PT-Only arcs
        total_pt_bike_arcs = df[df["Type"] == "PT+Bike"].shape[0]  # Total PT+Bike arcs
        used_pt_only_arcs = df[(df["Type"] == "PT-Only") & (df["Flow (q)"] >= 1)].shape[0]  # PT-Only arcs with flow > 0
        used_pt_bike_arcs = df[(df["Type"] == "PT+Bike") & (df["Flow (q)"] >= 1)].shape[0]  # PT+Bike arcs with flow > 0
        relax_difs_max = df[(df["Type"] == "PT+Bike")]["Relaxation Difference"].max()
        relax_difs_min = df[(df["Type"] == "PT+Bike")]["Relaxation Difference"].min()
        z_values_max = round(df[(df["Type"] == "PT+Bike")]["z value"].max(), 2)
        z_values_min = round(df[(df["Type"] == "PT+Bike")]["z value"].min(), 2)
        price_max= df[(df["Type"] == "PT+Bike")]["Bike Price"].max()
        price_min = df[(df["Type"] == "PT+Bike") & (df["Bike Price"] > 0.1)]["Bike Price"].min()
        # Append to summary list
        all_summaries_real_samples.append({
            "Instance Name": instance_name,
            "Solution Time (s)": solution_time,
            "Obj Value": Obj_Value,
            "Gap (%)": Gap,
            "Relaxation Difference max": relax_difs_max,
            "Relaxation Difference min": relax_difs_min,
            "z_max": z_values_max,
            "z_min": z_values_min,            
            "p_max": price_max,
            "p_min": price_min,
            "Number of OD Pairs": num_od_pairs,
            "Total Demand": total_demand,
            "Total Flow (Car)": total_flow_car,
            "Total Flow (PT Only)": total_flow_pt_only,
            "Total Flow (Multimodal)": total_flow_pt_bike,
            "Avg Bike Price (for all offered arcs)": avg_bike_price_pt_bike_total,
            "Avg Bike Price (Multimodal, Flow ≥ 1)": avg_bike_price_pt_bike,
            "Avg Number of Transfers (PT Only)": avg_num_transfers_pt_only,
            "Avg Number of Transfers (Multimodal)": avg_num_transfers_pt_bike,
            "Total PT Only Arcs": total_pt_only_arcs,
            "Used PT Only Arcs (Flow >= 1)": used_pt_only_arcs,
            "Total Multimodal Arcs": total_pt_bike_arcs,
            "Used Multimodal Arcs (Flow >= 1)": used_pt_bike_arcs
        })

# Convert to DataFrame and Print
summary_df_real_samples = pd.DataFrame(all_summaries_real_samples)
# 1) Compute usage ratios
summary_df_real_samples["PT Only Arcs Usage Ratio"] = (
    summary_df_real_samples["Used PT Only Arcs (Flow >= 1)"] / (summary_df_real_samples["Total PT Only Arcs"] + summary_df_real_samples["Total Multimodal Arcs"])
)
summary_df_real_samples["Multimodal Arcs Usage Ratio"] = (
    summary_df_real_samples["Used Multimodal Arcs (Flow >= 1)"] / (summary_df_real_samples["Total PT Only Arcs"] + summary_df_real_samples["Total Multimodal Arcs"])
)

# Melt into long form
usage_cols = ["PT Only Arcs Usage Ratio", "Multimodal Arcs Usage Ratio"]
df_usage = summary_df_real_samples.melt(
    id_vars=["Instance Name"],
    value_vars=usage_cols,
    var_name="Arc Type",
    value_name="Usage Ratio"
)
display(summary_df_real_samples.style.set_caption("Summary of Solved Instances From Real Network with L=7 (travel time is not integer) "))



